# Manejo de tools con agentes
## Carga de variables de entorno para obtener keys

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Prueba de uso

In [3]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openrouter/free",
    temperature=1,
)
response = llm.invoke("Hola, como estas?")
response.text

'¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú qué tal? ¿En qué puedo ayudarte hoy?'

In [10]:
from langchain_core.tools import tool
import requests

@tool("get_productos",description="Get the products that the store offers and the price of each one filter by price")
def get_products():
    """Busca productos en la API de Platzi"""
    # Conexión a API
    url = "https://api.escuelajs.co/api/v1/products"
    response = requests.get(url)
    products = response.json()
    return "".join([f"{product['title']} - ${product['price']}\n" for product in products])

In [11]:
get_products.invoke({})

'Majestic Mountain Graphic T-Shirt - $44\nClassic Red Pullover Hoodie - $10\nClassic Grey Hooded Sweatshirt - $90\nClassic Black Hooded Sweatshirt - $79\nClassic Comfort Fit Joggers - $25\nClassic Comfort Drawstring Joggers - $79\nClassic Red Jogger Sweatpants - $98\nClassic Navy Blue Baseball Cap - $61\nClassic Blue Baseball Cap - $86\nClassic Red Baseball Cap - $35\nClassic Black Baseball Cap - $58\nClassic Olive Chino Shorts - $84\nClassic High-Waisted Athletic Shorts - $43\nClassic White Crew Neck T-Shirt - $39\nClassic White Tee - Timeless Style and Comfort - $73\nClassic Black T-Shirt - $35\nSleek White & Orange Wireless Gaming Controller - $69\nSleek Wireless Headphone & Inked Earbud Set - $44\nSleek Comfort-Fit Over-Ear Headphones - $28\nEfficient 2-Slice Toaster - $48\nSleek Wireless Computer Mouse - $10\nSleek Modern Laptop with Ambient Lighting - $43\nSleek Modern Laptop for Professionals - $97\nStylish Red & Silver Over-Ear Headphones - $39\nSleek Mirror Finish Phone Case -

## Clima por ciudad checado con LLM

In [12]:
@tool("get_weather_by_city", description="Get coordinates for a given city")
def get_weather_by_city(city: str):
    """Return coordinates by city"""
    import requests
    from urllib.parse import quote

    url = f"https://geocoding-api.open-meteo.com/v1/search?name={quote(city)}&count=1&language=es&format=json"
    response = requests.get(url)
    data = response.json()

    if "results" not in data or not data["results"]:
        return []

    result = data["results"][0]
    latitude = result["latitude"]
    longitude = result["longitude"]

    return [latitude, longitude]

In [13]:
@tool("check_weather", description="Get current weather for a specified location (city name or coordinates)")
def check_weather(latitude="-40.157", longitude="-71.353"):
    "Get weather for a specified location"
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    weather = response.json()

    return weather

In [15]:
import json

[lat, lon] = get_weather_by_city.invoke("Lima")

result = check_weather.invoke(dict(latitude=lat, longitude=lon))


# print(json.dumps(result, indent=4, ensure_ascii=False))

cw = result["current_weather"]

formatted = (
    f"Current Weather Report\n"
    f"-----------------------\n"
    f"Temperature: {cw['temperature']} °C\n"
    f"Windspeed: {cw['windspeed']} km/h\n"
    f"Wind Direction: {cw['winddirection']}°\n"
    f"Is Day: {cw['is_day']}\n"
    f"Weather Code: {cw['weathercode']}\n"
)

print(formatted)

Current Weather Report
-----------------------
Temperature: 25.5 °C
Windspeed: 13.2 km/h
Wind Direction: 183°
Is Day: 1
Weather Code: 1



In [16]:
@tool("get_weather", description="Get weather for a city")
def get_weather(city: str) -> str:
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1")
    data = response.json()
    latitude = data["results"][0]["latitude"]
    longitude = data["results"][0]["longitude"]
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()
    response = f"The weather in {city} is {data["current_weather"]["temperature"]} C with {data["current_weather"]["windspeed"]} km/h of speed "
    return response



get_weather.invoke({"city": "quito"})

'The weather in quito is 12.8 C with 5.5 km/h of speed '